<a href="https://colab.research.google.com/github/francianerod/SojaMonitor-AI/blob/main/02_valor_do_modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
02 - VALOR DO MODELO sobre a regra de calculo dos 20,6 mm
SojaMonitor AI | Base: serie de Dourados/MS (Embrapa Agropecuaria Oeste), 1979-2023
Base: serie diaria de Dourados/MS (Embrapa Agropecuaria Oeste), 1979-2023

PREMISSA: com a janela de 5 dias completa, a classificacao e' aritmetica
(somar e comparar com 20,6 mm) e NAO precisa de modelo. Este script testa as
quatro situacoes em que a soma nao pode ser feita e o modelo passa a ter funcao:

  TAREFA A - ANTECIPACAO ....... prever hoje se a janela fechara em estiagem
                                 daqui a k dias (a soma ainda nao existe)
  TAREFA B - DADO FALTANTE ..... classificar quando faltam dias na janela
                                 (a soma e' impossivel)
  TAREFA C - RECALIBRACAO ...... derivar o limiar local de outra regiao
                                 (o 20,6 mm vale para Dourados)
  TAREFA D - ESPACIAL .......... estimar a condicao de area sem estacao
                                 (exige varias estacoes; estrutura pronta)

Cada tarefa e' comparada a um BASELINE sem modelo. O modelo so' se justifica
onde supera esse baseline.

Protocolo: treino ate 31/08/2021; validacao nas safras 2021/22 e 2022/23.
Requer: pandas, numpy, scikit-learn
"""
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             precision_score, recall_score, confusion_matrix)

CAMINHO = '/content/cpao_oficial_dados_1979_2023.csv'
LIMIAR = 20.6
CORTE_TREINO = '2021-08-31'
SAFRAS = {
    'safra ruim 2021/22': ('2021-09-01', '2022-03-31'),
    'safra boa  2022/23': ('2022-09-01', '2023-03-31'),
}


# --------------------------------------------------------------- base
def carregar(caminho=CAMINHO):
    """Serie diaria reindexada por data, com a janela movel de 5 dias."""
    df = pd.read_csv(caminho, sep=';')
    df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
    df = df.dropna().sort_values('data')
    df = df.set_index('data').asfreq('D').reset_index()

    df['acumulado5dias'] = df['chuva'].rolling(5, min_periods=5).sum()
    df['estiagem'] = (df['acumulado5dias'] < LIMIAR).astype('Int64')
    df.loc[df['acumulado5dias'].isna(), 'estiagem'] = pd.NA

    df['mes'] = df['data'].dt.month
    df['dia_do_ano'] = df['data'].dt.dayofyear
    # sazonalidade sem descontinuidade em 31/12
    ang = 2 * np.pi * df['dia_do_ano'] / 365.25
    df['sazonal_sen'], df['sazonal_cos'] = np.sin(ang), np.cos(ang)
    return df


def periodo(df, ini, fim):
    return df[(df['data'] >= ini) & (df['data'] <= fim)]


def linha(nome, y, p, pr=None):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    auc = roc_auc_score(y, pr) if pr is not None and len(set(y)) > 1 else float('nan')
    print(f"    {nome:24s} ACC {accuracy_score(y, p) * 100:5.1f}%  "
          f"BAL {balanced_accuracy_score(y, p) * 100:5.1f}%  "
          f"AUC {auc:.3f}  Prec {precision_score(y, p, zero_division=0):.2f}  "
          f"Sens {recall_score(y, p, zero_division=0):.2f}  "
          f"|  alarmes falsos {fp:3d}  perdidos {fn:3d}")


def floresta():
    return RandomForestClassifier(random_state=42, n_estimators=300, max_depth=12,
                                  min_samples_leaf=3, class_weight='balanced', n_jobs=-1)


# ============================================================ TAREFA A
def tarefa_antecipacao(df, k=3):
    """No dia t, prever se a janela de 5 dias encerrada em t+k estara' em estiagem.

    Nenhuma informacao posterior a t entra nas variaveis: a soma que decide o
    rotulo ainda nao aconteceu. Baseline = persistencia (repetir o estado de hoje).
    """
    print(f"\n{'=' * 92}\nTAREFA A — ANTECIPACAO DE {k} DIAS "
          f"(a janela alvo ainda nao fechou)\n{'=' * 92}")

    d = df.copy()
    for lag in range(0, 5):                      # chuva dos ultimos 5 dias
        d[f'chuva_l{lag}'] = d['chuva'].shift(lag)
    d['chuva_3d'] = d['chuva'].rolling(3, min_periods=3).sum()
    d['chuva_10d'] = d['chuva'].rolling(10, min_periods=10).sum()
    d['chuva_30d'] = d['chuva'].rolling(30, min_periods=30).sum()
    d['dias_secos_10'] = (d['chuva'] < 1).rolling(10, min_periods=10).sum()
    d['temp_5d'] = d['Tmedia'].rolling(5, min_periods=5).mean()
    d['estado_hoje'] = d['estiagem']             # o que a regra diz agora
    d['alvo'] = d['estiagem'].shift(-k)          # o que acontecera' em t+k

    F = ([f'chuva_l{l}' for l in range(5)] +
         ['chuva_3d', 'chuva_10d', 'chuva_30d', 'dias_secos_10',
          'Tmedia', 'temp_5d', 'estado_hoje', 'sazonal_sen', 'sazonal_cos'])
    d = d.dropna(subset=F + ['alvo'])

    tr = d[d['data'] <= CORTE_TREINO]
    m = floresta().fit(tr[F].values, tr['alvo'].astype(int).values)

    for nome, (ini, fim) in SAFRAS.items():
        v = periodo(d, ini, fim)
        if v.empty:
            continue
        y = v['alvo'].astype(int).values
        pr = m.predict_proba(v[F].values)[:, 1]
        print(f"\n  {nome}  (n={len(v)}, estiagem em t+{k} = {y.mean() * 100:.0f}% dos dias)")
        linha('baseline persistencia', y, v['estado_hoje'].astype(int).values,
              v['estado_hoje'].astype(float).values)
        linha('modelo (corte 0,50)', y, (pr >= .50).astype(int), pr)
        linha('modelo (corte 0,35)', y, (pr >= .35).astype(int), pr)


# ============================================================ TAREFA B
def tarefa_dado_faltante(df, faltando=2, seed=7):
    """Classificar a janela de 5 dias quando 'faltando' dias nao foram medidos.

    Baseline = extrapolacao proporcional: soma o que existe e projeta para 5 dias.
    """
    print(f"\n{'=' * 92}\nTAREFA B — JANELA INCOMPLETA "
          f"({faltando} de 5 dias sem medicao)\n{'=' * 92}")

    rng = np.random.default_rng(seed)
    chuva = df['chuva'].values
    n = len(df)
    soma_par, n_disp, max_disp, gap_recente = [], [], [], []

    for i in range(n):
        if i < 4 or np.isnan(chuva[i - 4:i + 1]).any():
            soma_par.append(np.nan); n_disp.append(np.nan)
            max_disp.append(np.nan); gap_recente.append(np.nan)
            continue
        jan = chuva[i - 4:i + 1].copy()
        fora = rng.choice(5, size=faltando, replace=False)
        vis = np.delete(jan, fora)
        soma_par.append(vis.sum())
        n_disp.append(5 - faltando)
        max_disp.append(vis.max())
        gap_recente.append(1.0 if 4 in fora else 0.0)   # falta o dia mais recente?

    d = df.copy()
    d['soma_parcial'] = soma_par
    d['n_disponiveis'] = n_disp
    d['max_disponivel'] = max_disp
    d['falta_dia_recente'] = gap_recente
    d['media_disponivel'] = d['soma_parcial'] / d['n_disponiveis']
    d['chuva_30d'] = d['chuva'].rolling(30, min_periods=30).sum()
    d['temp_5d'] = d['Tmedia'].rolling(5, min_periods=5).mean()

    F = ['soma_parcial', 'media_disponivel', 'max_disponivel', 'falta_dia_recente',
         'chuva_30d', 'Tmedia', 'temp_5d', 'sazonal_sen', 'sazonal_cos']
    d = d.dropna(subset=F + ['estiagem'])

    tr = d[d['data'] <= CORTE_TREINO]
    m = floresta().fit(tr[F].values, tr['estiagem'].astype(int).values)

    for nome, (ini, fim) in SAFRAS.items():
        v = periodo(d, ini, fim)
        if v.empty:
            continue
        y = v['estiagem'].astype(int).values
        extrap = (v['media_disponivel'].values * 5 < LIMIAR).astype(int)
        pr = m.predict_proba(v[F].values)[:, 1]
        print(f"\n  {nome}  (n={len(v)}, estiagem real = {y.mean() * 100:.0f}% dos dias)")
        linha('baseline extrapolacao', y, extrap, extrap.astype(float))
        linha('modelo (corte 0,50)', y, (pr >= .50).astype(int), pr)
        linha('modelo (corte 0,35)', y, (pr >= .35).astype(int), pr)


# ============================================================ TAREFA C
def calibrar_limiar(serie_chuva, datas, meses_safra=(9, 10, 11, 12, 1, 2, 3)):
    """Deriva o limiar local pelo mesmo criterio da tese: media das janelas de
    5 dias no periodo de safra, descartando outliers pelo IQR."""
    s = pd.Series(serie_chuva).rolling(5, min_periods=5).sum()
    m = pd.Series(datas).dt.month.isin(meses_safra).values
    v = s[m].dropna()
    q1, q3 = v.quantile(.25), v.quantile(.75)
    iqr = q3 - q1
    return float(v[(v >= q1 - 1.5 * iqr) & (v <= q3 + 1.5 * iqr)].mean())


def tarefa_recalibracao(df):
    """O limiar nao e' universal: muda com o regime de chuva local e com o tempo."""
    print(f"\n{'=' * 92}\nTAREFA C — RECALIBRACAO DO LIMIAR\n{'=' * 92}")
    geral = calibrar_limiar(df['chuva'].values, df['data'])
    print(f"\n  serie completa 1979-2023 .......... {geral:.1f} mm   (limiar da tese: {LIMIAR} mm)")
    print("\n  por decada (mesma estacao, periodos diferentes):")
    for ini, fim in [('1980-01-01', '1989-12-31'), ('1990-01-01', '1999-12-31'),
                     ('2000-01-01', '2009-12-31'), ('2010-01-01', '2019-12-31'),
                     ('2014-01-01', '2023-12-31')]:
        d = periodo(df, ini, fim)
        L = calibrar_limiar(d['chuva'].values, d['data'])
        marca = '  <-- afasta-se de Dourados' if abs(L - geral) > 1.5 else ''
        print(f"    {ini[:4]}-{fim[:4]} .................... {L:5.1f} mm{marca}")
    print("\n  leitura: o criterio de calibracao e' reprodutivel, mas o valor do limiar"
          "\n  muda com o periodo e mudara' com a regiao. Aplicar 20,6 mm fora de Dourados"
          "\n  sem recalibrar introduz erro sistematico — por isso a recalibracao e' etapa"
          "\n  do produto, nao detalhe de implementacao.")


# ============================================================ TAREFA D
def tarefa_espacial(estacoes=None):
    """Estimar a condicao de uma area sem estacao a partir das estacoes vizinhas.

    Exige ao menos 3 estacoes com series simultaneas. Com uma unica estacao nao
    ha' como treinar nem validar: a estrutura fica pronta para a fase do projeto
    em que as 57 estacoes publicas de MS forem integradas.

    estacoes: dict {nome: DataFrame com colunas data, chuva, Tmedia, lat, lon}
    """
    print(f"\n{'=' * 92}\nTAREFA D — ESTIMATIVA PARA AREA SEM ESTACAO\n{'=' * 92}")
    if not estacoes or len(estacoes) < 3:
        print("\n  Nao executada: requer 3 ou mais estacoes com series simultaneas.")
        print("  Protocolo previsto (validacao leave-one-station-out):")
        print("    1. treinar com as estacoes vizinhas, escondendo uma estacao inteira;")
        print("    2. prever a estiagem da estacao escondida a partir das vizinhas,")
        print("       ponderadas por distancia;")
        print("    3. comparar com o baseline 'copiar a estacao mais proxima';")
        print("    4. repetir para cada estacao e reportar o erro medio por distancia.")
        print("\n  O ganho do modelo aparece onde a estacao mais proxima esta' longe")
        print("  o bastante para que copiar deixe de funcionar.")
        return
    print("\n  (execucao com multiplas estacoes — a implementar na fase 2)")


# ============================================================ main
def main():
    df = carregar()
    print(f"Serie: {df['data'].min().date()} a {df['data'].max().date()}  "
          f"({len(df)} dias, {int(df['chuva'].isna().sum())} sem medicao)")
    print(f"Regra de referencia: acumulado de 5 dias < {LIMIAR} mm = estiagem da cultura")
    print("Com a janela completa a regra acerta 100% por construcao — o modelo so' "
          "\ne' avaliado abaixo nas situacoes em que a regra nao pode ser aplicada.")

    tarefa_antecipacao(df, k=3)
    tarefa_dado_faltante(df, faltando=2)
    tarefa_recalibracao(df)
    tarefa_espacial()


if __name__ == '__main__':
    main()

Serie: 1979-06-01 a 2023-12-31  (16285 dias, 4 sem medicao)
Regra de referencia: acumulado de 5 dias < 20.6 mm = estiagem da cultura
Com a janela completa a regra acerta 100% por construcao — o modelo so' 
e' avaliado abaixo nas situacoes em que a regra nao pode ser aplicada.

TAREFA A — ANTECIPACAO DE 3 DIAS (a janela alvo ainda nao fechou)

  safra ruim 2021/22  (n=212, estiagem em t+3 = 70% dos dias)
    baseline persistencia    ACC  74.1%  BAL  68.3%  AUC 0.683  Prec 0.81  Sens 0.83  |  alarmes falsos  29  perdidos  26
    modelo (corte 0,50)      ACC  79.7%  BAL  72.3%  AUC 0.704  Prec 0.82  Sens 0.91  |  alarmes falsos  29  perdidos  14
    modelo (corte 0,35)      ACC  80.2%  BAL  69.9%  AUC 0.704  Prec 0.80  Sens 0.95  |  alarmes falsos  35  perdidos   7

  safra boa  2022/23  (n=212, estiagem em t+3 = 50% dos dias)
    baseline persistencia    ACC  58.0%  BAL  58.0%  AUC 0.580  Prec 0.58  Sens 0.59  |  alarmes falsos  46  perdidos  43
    modelo (corte 0,50)      ACC  74.5%  B